# Knowledge Distillation - Training

<a target="_blank" href="https://colab.research.google.com/github/WholeNow/SmolLM-KD/blob/main/training.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Install dependencies and import libraries

In [ ]:
!pip install -q transformers datasets trl evaluate rouge_score bert_score matplotlib accelerate bitsandbytes

In [ ]:
# All imports
from datasets import load_dataset, Dataset, load_from_disk
from transformers import pipeline, AutoTokenizer, GenerationConfig, AutoModelForCausalLM
from tqdm.auto import tqdm
from transformers.pipelines.pt_utils import KeyDataset
from trl import SFTTrainer, SFTConfig
import evaluate
import math
import random
import os
import gc
import torch
import time

## Configuration

| Parameter | Description | Allowed Values |
| :--- | :--- | :--- |
| **TASK** | The NLP task to execute (determines dataset and prompt). | `"summarization"`, `"question_answering"` |
| **PROMPT_TYPE** | Type of prompt to use for the Teacher. | `"1 -> prompt with negations"`, `"2 -> prompt with direct questions"`, `"3 -> minimal prompt"` |
| **TEACHER_MODEL_ID** | The "large" model from which to distill labels. | HuggingFace ID (`"TinyLlama/TinyLlama-1.1B-Chat-v1.0"`) |
| **QUANTIZE_TEACHER** | If `True`, quantize the Teacher to 4-bit NF4. | `True` or `False` |
| **STUDENT_MODEL_ID** | The "small" model to train via SFT. | HuggingFace ID (`"HuggingFaceTB/SmolLM-135M"`) |
| **MAX_TRAIN_SAMPLES** | Maximum samples for Teacher generation. | `"all"`, or integer string (e.g., `"10000"`) |
| **MAX_TEST_SAMPLES** | Maximum samples for final evaluation. | `"all"`, or integer string (e.g., `"150"`) |
| **TEACHER_PIPELINE_BATCH_SIZE**| Batch size for Teacher inference. | Positive integer (e.g., `16`) |
| **STUDENT_BATCH_SIZE** | Batch size per device (Student training). | Positive integer (e.g., `4`) |
| **GRAD_ACCUMULATION** | Gradient accumulation steps for Student. | Positive integer (e.g., `4`) |
| **EPOCHS** | Number of training epochs. | Positive integer (e.g., `3`) |
| **LEARNING_RATE** | Maximum learning rate (Cosine scheduler). | Float (e.g., `1e-5`) |
| **MAX_SEQ_LENGTH** | Maximum sequence length in tokens for Student. | Positive integer (e.g., `1024`) |

In [ ]:
# ============================================================
#                     CONFIGURATION
# ============================================================

# TASK:
# "summarization" (knkarthick/samsum)
# "question_answering" (databricks/databricks-dolly-15k)
TASK = "summarization"

# PROMPT_TYPE:
# 1: prompt with negation
# 2: prompt with direct question
# 3: minimal prompt
PROMPT_TYPE = 3

# TEACHER: choose one of the models
# "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# "Qwen/Qwen2.5-1.5B-Instruct"
# "Qwen/Qwen3-4B-Instruct-2507"
TEACHER_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit NF4 quantization for the Teacher.
# True : NF4 4-bit via bitsandbytes
# False: standard float16
QUANTIZE_TEACHER = True

# STUDENT:
# "HuggingFaceTB/SmolLM-135M"
STUDENT_MODEL_ID = "HuggingFaceTB/SmolLM-135M"

# Number of samples for Teacher training (label generation)
# Use 'all' for the full dataset, or an integer e.g. 500
MAX_TRAIN_SAMPLES = '10000'

# Number of samples for final evaluation
# Use 'all' or an integer e.g. 100
MAX_TEST_SAMPLES = '150'

# Batch size for the Teacher pipeline
TEACHER_PIPELINE_BATCH_SIZE = 16

# Student training hyperparameters
STUDENT_BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 2
LEARNING_RATE = 1e-5
MAX_SEQ_LENGTH = 1024

# Output directories
BASE_OUTPUT_DIR = f"./results"
TEACHER_DATASET_DIR = os.path.join(BASE_OUTPUT_DIR, TASK, f"prompt_{PROMPT_TYPE}", "dataset_teacher")
STUDENT_OUTPUT_DIR  = os.path.join(BASE_OUTPUT_DIR, TASK, f"prompt_{PROMPT_TYPE}", "checkpoints")
STUDENT_FINAL_DIR   = os.path.join(BASE_OUTPUT_DIR, TASK, f"prompt_{PROMPT_TYPE}", "model")

## Utils

In [ ]:
def clear_memory():
    for var in ['generator', 'student_model', 'model', 'trainer']:
        if var in globals():
            del globals()[var]
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
def is_qwen3(model_id: str) -> bool:
    """True if the model ID belongs to the Qwen3 family (has thinking mode)."""
    return "qwen3" in model_id.lower()

In [ ]:
def build_messages(example):
    """
    Builds the message list (system + user) based on TASK and PROMPT_TYPE.
    Returns (messages, target, input_text).
    """
    if TASK == "summarization":
        if PROMPT_TYPE == 1:
            messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert assistant strictly dedicated to abstractive summarization. "
                    "You must extract the core event, problem, or decision from the conversation. "
                    "RULES: "
                    "1) Do NOT copy, repeat, or quote the dialogue. "
                    "2) Do NOT use dialogue format (e.g., 'Name:'). "
                    "3) Write exactly one or two sentences in the third person."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Dialogue:\n{example['dialogue']}\n\n"
                    "Task: Write a brief, third-person narrative summary describing what the people are doing or talking about."
                )
            }
            ]
        elif PROMPT_TYPE == 2:
            messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert assistant strictly dedicated to abstractive summarization. "
                    "Your task is to extract the core event, problem, or decision from the conversation. "
                    "You MUST adhere strictly to the following RULES: "
                    "1) Paraphrase the dialogue entirely in your own words. "
                    "2) Format the output strictly as standard continuous prose. "
                    "3) Write exactly one or two sentences in the third person. "
                    "You will be penalized if you fail to follow these formatting instructions."
                )
            },
            {
                "role": "user",
                "content": (
                    "###Instruction###\n"
                    "Write a brief, third-person narrative summary describing what the people are doing or talking about.\n\n"
                    "###Dialogue###\n"
                    f"{example['dialogue']}\n\n"
                    "###Summary###\n"
                )
            }
            ]
        elif PROMPT_TYPE == 3:
            messages = [
            {
                "role": "system",
                "content": "You are a summary expert."
            },
            {
                "role": "user",
                "content": (
                    "Summarize the following dialogue in a sentence\n" +
                    f"Dialogue: {example['dialogue']}\n"
                    "Summary:\n"
                )
            }
            ]
        return messages, example["summary"], example["dialogue"]



    elif TASK == "question_answering":
        if PROMPT_TYPE == 1:
            messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert assistant strictly dedicated to question answering. "
                    "You must answer the user's question accurately. If a context is provided, base your answer on it. "
                    "RULES: "
                    "1) Provide a clear and concise answer. "
                    "2) Do NOT add unnecessary conversational filler."
                )
            },
            {
                "role": "user",
                "content": (
                    (f"Context:\n{example.get('context', '')}\n\n" if example.get('context') else "") +
                    f"Question:\n{example['instruction']}\n\n"
                    "Task: Answer the question."
                )
            }
            ]
        elif PROMPT_TYPE == 2:
            messages = [
            {
                "role": "system",
                "content": (
                    "###Instruction###\n"
                    "You are an expert assistant strictly dedicated to question answering. "
                    "Your task is to answer the user's question accurately. "
                    "If a context is provided, you MUST base your answer solely on it. "
                    "RULES:\n"
                    "1) Provide a clear and concise answer.\n"
                    "2) Maintain strict focus and provide only the essential information. "
                    "You will be penalized for generating unnecessary conversational filler."
                )
            },
            {
                "role": "user",
                "content": (
                    (f"###Context###\n{example.get('context', '')}\n\n" if example.get('context') else "") +
                    f"###Question###\n{example['instruction']}\n\n"
                    "###Answer###\n"
                )
            }
            ]
        elif PROMPT_TYPE == 3:
            messages = [
                {
                    "role": "system",
                    "content": "You are a question answering expert."
                },
                {
                    "role": "user",
                    "content": (
                        "Answer the following question based on the provided context (if any):\n" +
                        (f"Context: {example.get('context', '')}\n" if example.get('context') else "") +
                        f"Question: {example['instruction']}\n"
                        "Answer:\n"
                    )
                }
            ]


        return messages, example["response"], messages[1]["content"]


def build_prompt(tokenizer, example):
    messages, target, input_text = build_messages(example)
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    # Qwen3 has thinking mode enabled by default; disabling it avoids <think>...</think>
    # blocks in the output, which would pollute the distillation pseudo-labels.
    if is_qwen3(TEACHER_MODEL_ID):
        kwargs["enable_thinking"] = False
    return {
        "prompt": tokenizer.apply_chat_template(messages, **kwargs),
        "target": target,
        "input_text": input_text
    }

## Distilled Dataset Generation (Teacher)
HuggingFace `Pipeline` with native batching to efficiently generate Teacher pseudo-labels and save them to disk.
If the file already exists, it is loaded directly.

In [ ]:
# ── VRAM/RAM cleanup ──
clear_memory()

# ── Prompt configuration ──────────────────
if TASK == "summarization":
    raw_dataset = load_dataset("knkarthick/samsum", split="train")
elif TASK == "question_answering":
    full_dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    raw_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)["train"]

# ── Teacher tokenizer initialization ────────────────────
tokenizer_teacher = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
if tokenizer_teacher.pad_token is None:
    tokenizer_teacher.pad_token = tokenizer_teacher.eos_token

# ── Sample limit ─────────────────────────────────────
if MAX_TRAIN_SAMPLES != 'all':
    raw_dataset = raw_dataset.select(range(min(int(MAX_TRAIN_SAMPLES), len(raw_dataset))))

print(f"Task: {TASK} | Teacher: {TEACHER_MODEL_ID} | Train samples: {len(raw_dataset)}")
print("\nDataset columns:", raw_dataset.column_names)

# Cap at 4096 to prevent models with huge context windows (e.g. Qwen3: 32768+)
# from exhausting all VRAM during Teacher filtering and inference
TEACHER_MAX_TOKENS = min(tokenizer_teacher.model_max_length or 2048, 4096)
TOKEN_MARGIN = 128  # equals max_new_tokens in gen_config

def filter_by_token_length(dataset, tokenizer, max_tokens, margin):
    """
    Filters out samples whose prompt exceeds (max_tokens - margin) tokens.
    Works on any dataset containing the original task fields
    (dialogue/summary for summarization, instruction/context/response for QA),
    because it rebuilds the prompt via build_prompt.

    Returns the filtered dataset and the number of discarded samples.
    """
    prompts_raw = [build_prompt(tokenizer, example) for example in dataset]
    valid_indices = []
    skipped = 0
    threshold = max_tokens - margin
    for i, p in enumerate(prompts_raw):
        n_tokens = len(tokenizer(p["prompt"], add_special_tokens=False)["input_ids"])
        if n_tokens <= threshold:
            valid_indices.append(i)
        else:
            skipped += 1
    if skipped:
        print(f"[WARN] {skipped} samples filtered because the prompt exceeds {threshold} tokens.")
    return dataset.select(valid_indices), skipped

# ── Check for existing dataset on disk ──────────────────────────────────────
if os.path.exists(TEACHER_DATASET_DIR):
    print(f"\nDataset found on disk: {TEACHER_DATASET_DIR}")
    print("Loading...")
    distilled_dataset = load_from_disk(TEACHER_DATASET_DIR)
    print(f"Samples loaded: {len(distilled_dataset)}")

    print("Applying token filter to loaded dataset...")
    distilled_dataset, n_skipped = filter_by_token_length(
        distilled_dataset, tokenizer_teacher, TEACHER_MAX_TOKENS, TOKEN_MARGIN
    )
    print(f"Dataset ready: {len(distilled_dataset)} valid samples ({n_skipped} discarded).")

else:
    raw_dataset, _ = filter_by_token_length(
        raw_dataset, tokenizer_teacher, TEACHER_MAX_TOKENS, TOKEN_MARGIN
    )
    prompts = [build_prompt(tokenizer_teacher, example) for example in raw_dataset]
    prompt_dataset = Dataset.from_dict({"prompt": [p["prompt"] for p in prompts]})

    print("\nStarting pseudo-label generation...")
    teacher_outputs = []

    # 1. Generation config (max 128 tokens, greedy decoding for reproducibility)
    # NOTE: return_full_text does NOT go in GenerationConfig (it is ignored there);
    # it must be passed directly to the pipeline in the loop, along with generation_config.
    gen_config = GenerationConfig(
        max_new_tokens=128,
        do_sample=False,
        max_length=None,
        num_beams=1,
    )

    # 2. Teacher pipeline initialization
    if QUANTIZE_TEACHER:
        from transformers import BitsAndBytesConfig
        _bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        generator = pipeline(
            "text-generation",
            model=TEACHER_MODEL_ID,
            model_kwargs={"quantization_config": _bnb_config},
            device_map="auto",
            batch_size=TEACHER_PIPELINE_BATCH_SIZE,
        )
    else:
        generator = pipeline(
            "text-generation",
            model=TEACHER_MODEL_ID,
            dtype=torch.float16,
            device_map="auto",
            batch_size=TEACHER_PIPELINE_BATCH_SIZE,
        )
    generator.model.generation_config.max_length = None  # Clear native limit to avoid conflicts

    # 3. Pseudo-label generation with progress bar.
    # return_full_text=False ensures generated_text contains only the new tokens
    # (Teacher's response), without the prompt prepended.
    # Works correctly with both TinyLlama (<|assistant|>) and Qwen (<|im_start|>assistant).
    for out in tqdm(
        generator(
            KeyDataset(prompt_dataset, "prompt"),
            generation_config=gen_config,
            return_full_text=False,
        ),
        total=len(prompts),
        desc=f"Distillation for {TASK}"
    ):
        clean_output = out[0]['generated_text'].strip()
        teacher_outputs.append(clean_output)

    # 4. Add pseudo-labels to dataset and save
    match TASK:
        case "question_answering":
            column_name = "teacher_answer"
        case "summarization":
            column_name = "teacher_summary"
        case _:
            raise ValueError(f"Task {TASK} not supported.")

    distilled_dataset = raw_dataset.add_column(column_name, teacher_outputs)
    distilled_dataset.save_to_disk(TEACHER_DATASET_DIR)
    print("Dataset saved successfully.")

## Student Training
Fine-tuning the Student on Teacher-generated pseudo-labels via `SFTTrainer`.

Steps performed in this section:
1. Map the dataset to the message format (system/user/assistant).
2. Inject the chat template with standard stop tokens for correct generation termination.
3. Launch supervised fine-tuning (SFT).

In [ ]:
# Map to native Prompt-Completion format
def format_example(example):
    messages, _, _ = build_messages(example)

    if TASK == "summarization":
        assistant_msg = example['teacher_summary']
    elif TASK == "question_answering":
        assistant_msg = example['teacher_answer']

    return {"messages": messages + [{"role": "assistant", "content": assistant_msg}]}

In [ ]:
# ── VRAM/RAM cleanup ──
clear_memory()

# ── Student tokenizer initialization ───────────────────
tokenizer_student = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)
if tokenizer_student.pad_token is None:
    tokenizer_student.pad_token = tokenizer_student.eos_token


# ── Add special tokens <|im_start|> and <|im_end|> ────────────────────────
special_tokens_to_add = []
if "<|im_start|>" not in tokenizer_student.vocab:
    special_tokens_to_add.append("<|im_start|>")
if "<|im_end|>" not in tokenizer_student.vocab:
    special_tokens_to_add.append("<|im_end|>")

if special_tokens_to_add:
    tokenizer_student.add_special_tokens({"additional_special_tokens": special_tokens_to_add})
    print(f"Special tokens added: {special_tokens_to_add}")


# ── Forced ChatML template injection ──────────────────────
# Template structure:
#
#   <|im_start|>system\n
#   [system content]<|im_end|>\n
#   <|im_start|>user\n
#   [user content]<|im_end|>\n
#   <|im_start|>assistant\n
#   {% generation %}[assistant response]<|im_end|>\n{% endgeneration %}
#
# The {% generation %} markers delimit the tokens on which loss is computed.
# When TRL calls apply_chat_template(..., return_assistant_tokens_mask=True),
# it receives a binary mask: 1 for tokens inside {% generation %}, 0 for others.
#
# WHY <|im_end|> and NOT {{ eos_token }}:
#   - <|im_end|> has a precise semantic meaning: "end of assistant turn"
#   - </s> (native eos) is ambiguous: means both "end of turn" and "end of sequence"
#   - By setting eos_token="<|im_end|>" in SFTConfig, the model learns to stop there
#   - This is the ChatML standard used by Qwen, Mistral, etc.
tokenizer_student.chat_template = (
    "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n"
        "{% if message['role'] == 'assistant' %}"
            "{% generation %}"
            "{{ message['content'] }}<|im_end|>\n"
            "{% endgeneration %}"
        "{% else %}"
            "{{ message['content'] }}<|im_end|>\n"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
        "<|im_start|>assistant\n"
    "{% endif %}"
)


# ── Student model loading ───────────────────
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    dtype=torch.float32,
    device_map="auto"
)

if special_tokens_to_add:
    model.resize_token_embeddings(len(tokenizer_student))
    print(f"Embedding matrix resized to {len(tokenizer_student)} tokens")

# If the dataset is not in memory, look for it on disk; otherwise raise an error
if 'distilled_dataset' in globals():
    pc_dataset = distilled_dataset.map(format_example, remove_columns=distilled_dataset.column_names)
elif os.path.exists(TEACHER_DATASET_DIR):
    distilled_dataset = load_from_disk(TEACHER_DATASET_DIR)
    pc_dataset = distilled_dataset.map(format_example, remove_columns=distilled_dataset.column_names)
else:
    raise ValueError(f"Dataset {TEACHER_DATASET_DIR} not found. Make sure Teacher generation completed successfully.")

# ── Validation split to monitor overfitting ─────────────────────────────────
split      = pc_dataset.train_test_split(test_size=0.05, seed=42)
train_data = split["train"]
eval_data  = split["test"]

print(f"\nDataset: {len(train_data)} train / {len(eval_data)} eval")

# SFTConfig configuration
training_args = SFTConfig(
    output_dir=STUDENT_OUTPUT_DIR,
    per_device_train_batch_size=STUDENT_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    logging_first_step=True,
    num_train_epochs=EPOCHS,
    fp16=True,
    optim="adamw_torch_fused",
    report_to="none",
    max_length=MAX_SEQ_LENGTH,

    # ── ASSISTANT-ONLY LOSS ────────────────────────────────────────────────
    # assistant_only_loss=True is the correct flag for datasets with a "messages" column.
    # completion_only_loss=True (original) is for {"prompt": ..., "completion": ...} datasets.
    # With the messages dataset and completion_only_loss, the loss is computed
    # over the entire sequence (system + user + assistant), diluting the signal.
    assistant_only_loss=True,

    # ── EOS TOKEN ALIGNED TO TEMPLATE ──────────────────────────────────────
    # The chat template uses <|im_end|> as the assistant stop token.
    # SFTConfig needs to know this to correctly align the loss mask.
    # Without this, TRL looks for </s> as end-of-response but the template has <|im_end|>.
    eos_token="<|im_end|>",

    # Save the best checkpoint based on eval loss
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    dataset_kwargs={
        "skip_prepare_dataset": False
    },
)

# Run training
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer_student,
)

print("Starting training...")
trainer.train()

# Save
trainer.save_model(STUDENT_FINAL_DIR)
tokenizer_student.save_pretrained(STUDENT_FINAL_DIR)
print(f"Model and tokenizer saved to {STUDENT_FINAL_DIR}.")

 ## Evaluation
Comparison of **Teacher**, **Student Baseline (zero-shot)**, and **Student Distilled**.
Metrics: ROUGE-1, ROUGE-2, ROUGE-L, BERTScore-F1, Perplexity, Latency/Token, Parameters.

In [ ]:
def evaluate_model(model_path, chat_template=False, quantize=False):
    """
    Evaluates a model on ROUGE, BERTScore, Perplexity, and Latency.

    chat_template=True → injects the ChatML template (for distilled Student and baseline).
    chat_template=False → uses the model's native template (for Teacher).
    quantize=True       → loads the model in 4-bit NF4 (required for Teacher >= 2B on T4).

    NOTE on Perplexity:
      Computed by masking the prompt (label=-100 on prompt tokens).
      Measures how "surprised" the model is by the target, given the prompt.
      Lower = better.

    NOTE on generation:
      stop_tokens includes both eos_token_id and <|im_end|> if present in the vocabulary.
      This ensures the model stops correctly with both templates.
    """
    print(f"\nEvaluating: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if chat_template and tokenizer.chat_template is None:
        tokenizer.chat_template = (
            "{% for message in messages %}"
                "<|im_start|>{{ message['role'] }}\n"
                "{% if message['role'] == 'assistant' %}"
                    "{% generation %}"
                    "{{ message['content'] }}<|im_end|>\n"
                    "{% endgeneration %}"
                "{% else %}"
                    "{{ message['content'] }}<|im_end|>\n"
                "{% endif %}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
                "<|im_start|>assistant\n"
            "{% endif %}"
        )

    if quantize:
        from transformers import BitsAndBytesConfig
        _bnb_eval = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_path, quantization_config=_bnb_eval, device_map="auto"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32).to(device)
    model.eval()
    model.generation_config.max_length = None  # Clear native limit to avoid conflicts

    predictions, references = [], []
    total_loss, total_time, total_gen_tokens = 0.0, 0.0, 0

    # Stop tokens: stop generation on both </s> and <|im_end|> if available
    stop_tokens = [tokenizer.eos_token_id]
    if "<|im_end|>" in tokenizer.vocab:
        im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        if im_end_id != tokenizer.eos_token_id:
            stop_tokens.append(im_end_id)


    for sample in tqdm(test_raw, desc=f"Eval {model_path}"):
        messages, target, _ = build_messages(sample)

        _tpl_kwargs = {"tokenize": False, "add_generation_prompt": True}
        if is_qwen3(model_path):
            _tpl_kwargs["enable_thinking"] = False
        prompt = tokenizer.apply_chat_template(messages, **_tpl_kwargs)
        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
        prompt_len = inputs["input_ids"].shape[1]

        # Perplexity computation: full sequence (prompt + target + eos)
        full_text = prompt + target + tokenizer.eos_token
        full_inputs = tokenizer(full_text, return_tensors="pt").to(device)

        prompt_only_inputs = tokenizer(prompt, return_tensors="pt").to(device)
        prompt_len_full = prompt_only_inputs["input_ids"].shape[1]


        # Mask prompt tokens in labels
        labels = full_inputs["input_ids"].clone()
        labels[0, :prompt_len_full] = -100  # Mask prompt: loss only on target

        with torch.no_grad():
            loss_out = model(**full_inputs, labels=labels)
            total_loss += loss_out.loss.item()

            # Generation for ROUGE/BERTScore
            t0 = time.time()
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=stop_tokens
            )
            total_time += time.time() - t0

        gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        total_gen_tokens += len(gen_tokens)
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        predictions.append(gen_text)
        references.append(target)

    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    bert_res = bert_metric.compute(predictions=predictions, references=references, lang="en")
    avg_bert_f1 = sum(bert_res["f1"]) / len(bert_res["f1"])
    avg_loss = total_loss / len(test_raw)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    ms_per_tok = (total_time * 1000) / total_gen_tokens if total_gen_tokens > 0 else 0
    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    del model
    torch.cuda.empty_cache()

    return {
        "Perplexity":        perplexity,
        "ROUGE-1":           rouge_res["rouge1"],
        "ROUGE-2":           rouge_res["rouge2"],
        "ROUGE-L":           rouge_res["rougeL"],
        "BERTScore-F1":      avg_bert_f1,
        "Latency/Token (ms)": ms_per_tok,
        "Parameters (M)":    params_m,
        "predictions":       predictions,
    }

In [ ]:
# ── VRAM/RAM cleanup to avoid OOM on restart/interruption ──
clear_memory()

# 1. Metrics and hardware setup
device = "cuda" if torch.cuda.is_available() else "cpu"
rouge_metric = evaluate.load("rouge")
bert_metric = evaluate.load("bertscore")

# Load original test dataset
if TASK == "summarization":
    test_raw = load_dataset("knkarthick/samsum", split="test")
elif TASK == "question_answering":
    full_dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    test_raw = full_dataset.train_test_split(test_size=0.1, seed=42)["test"]

if MAX_TEST_SAMPLES != 'all':
    test_raw = test_raw.select(range(min(int(MAX_TEST_SAMPLES), len(test_raw))))

print(f"Test samples: {len(test_raw)}")

# 2. Run comparisons
metrics_distilled = evaluate_model(STUDENT_FINAL_DIR,  chat_template=False, quantize=False)
metrics_baseline  = evaluate_model(STUDENT_MODEL_ID,   chat_template=True,  quantize=False)
metrics_teacher   = evaluate_model(TEACHER_MODEL_ID,   chat_template=False, quantize=QUANTIZE_TEACHER)

# 3. Print comparative results
print("\n=== COMPARATIVE RESULTS ===")
for name, m in [("Teacher", metrics_teacher), ("Student Baseline", metrics_baseline), ("Student Distilled", metrics_distilled)]:
    print(f"\n{name}:")
    for k, v in m.items():
        if k != "predictions":
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

### Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_metrics = ["Perplexity", "ROUGE-L", "BERTScore-F1", "Latency/Token (ms)"]
models       = ["Teacher", "Student Baseline", "Student Distilled"]
all_results  = [metrics_teacher, metrics_baseline, metrics_distilled]
colors       = ["steelblue", "lightcoral", "mediumseagreen"]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(14, 5))
fig.suptitle(f"Knowledge Distillation — Task: {TASK}", fontsize=14, fontweight="bold")

for ax, metric in zip(axes, plot_metrics):
    vals = [r[metric] for r in all_results]
    bars = ax.bar(models, vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_title(metric, fontsize=11)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in models], fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"kd_results_{TASK}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved as kd_results_{TASK}.png")

### Qualitative Inspection
Print 3 random examples comparing the three models.

In [ ]:
# Load distilled model in fp32 for qualitative inspection
tok_qual   = AutoTokenizer.from_pretrained(STUDENT_FINAL_DIR)
model_qual = AutoModelForCausalLM.from_pretrained(STUDENT_FINAL_DIR, dtype=torch.float32, device_map="auto")

samples = random.sample(list(test_raw), min(3, len(test_raw)))

stop_tokens = [tok_qual.eos_token_id]
if "<|im_end|>" in tok_qual.vocab:
    im_end_id = tok_qual.convert_tokens_to_ids("<|im_end|>")
    if im_end_id != tok_qual.eos_token_id:
        stop_tokens.append(im_end_id)

print("=== QUALITATIVE OUTPUT INSPECTION ===\n")
for i, sample in enumerate(samples, 1):
    messages, target, input_text = build_messages(sample)
    prompt = tok_qual.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok_qual(prompt, return_tensors="pt").to(model_qual.device)

    with torch.no_grad():
        outputs = model_qual.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=tok_qual.eos_token_id,
            eos_token_id=stop_tokens
        )
    gen_text = tok_qual.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

    print(f"--- SAMPLE {i} ---")
    print(f"INPUT:\n{input_text.strip()}\n")
    print(f"IDEAL TARGET (Human):\n{target.strip()}\n")
    print(f"DISTILLED STUDENT GENERATION:\n{gen_text.strip()}\n")
    print("STUDENT BASELINE prediction:", metrics_baseline["predictions"][test_raw.to_list().index(sample)] if hasattr(test_raw, 'to_list') else "(run with index)")
    print("=" * 60 + "\n")